## Model 1: LightGBM 




In [13]:
pip install optuna


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install lightgbm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna
import lightgbm as lgb


In [16]:
# importing data
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/train.csv')
irrigation.head(20)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
5,5,Sandy,5.09,24.70,1.28,0.48,12.21,92.35,696.03,9.11,...,Sugarcane,Flowering,Kharif,Sprinkler,River,0.81,No,86.56,South,Medium
6,6,Silt,7.53,49.67,1.44,1.62,14.02,61.65,889.39,8.44,...,Potato,Flowering,Zaid,Sprinkler,Rainwater,13.32,No,14.05,East,Medium
7,7,Loamy,7.56,48.61,0.38,1.31,22.78,61.53,708.82,9.96,...,Rice,Sowing,Zaid,Sprinkler,Reservoir,5.63,Yes,110.99,East,Low
8,8,Silt,6.02,53.01,0.90,0.49,20.55,61.30,1536.36,10.50,...,Potato,Flowering,Kharif,Rainfed,River,8.17,Yes,36.37,West,Low
9,9,Silt,7.39,41.91,0.58,0.78,39.25,85.52,281.76,8.86,...,Wheat,Vegetative,Kharif,Rainfed,Reservoir,5.41,No,71.92,Central,High


In [17]:
# splitting data into train and test sets
X = irrigation.drop(['Irrigation_Need', 'id'], axis=1)
y = irrigation['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42) 


In [18]:
# Identify all columns in your training data that are of type 'object' (strings)
categorical_cols = X_train.select_dtypes(include=['object']).columns
# Convert those object columns to 'category' type in both train and test sets
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col]  = X_test[col].astype('category')
print("Categorical columns converted to 'category' data type:")
print(list(categorical_cols))

Categorical columns converted to 'category' data type:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [19]:
lgb_base = lgb.LGBMClassifier(random_state=42)
lgb_base.fit(X_train, y_train)
y_pred_base = lgb_base.predict(X_test)
print("Base Model Accuracy:", accuracy_score(y_test, y_pred_base))
print("\nBase Model Classification Report:")
print(classification_report(y_test, y_pred_base))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050411 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2702
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
Base Model Accuracy: 0.9848015873015873

Base Model Classification Report:
              precision    recall  f1-score   support

        High       0.95      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.97      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000



## Tuning with GridSearchCV


In [20]:
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'num_leaves': [15, 31]}

lgbm_grid = lgb.LGBMClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=lgbm_grid,
    param_grid=param_grid,
    cv=3,                 # 3-fold cross validation
    scoring='accuracy',
    n_jobs=-1,            
    verbose=1
)
grid_search.fit(X_train, y_train)
print("Best GridSearchCV Parameters:", grid_search.best_params_)
y_pred_grid = grid_search.best_estimator_.predict(X_test)
print("GridSearchCV Accuracy:", accuracy_score(y_test, y_pred_grid))

y_pred_grid = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred_grid))
print("\n")


Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009490 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 336000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400840
[LightGBM] [Info] Start training from score -0.532436
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.129880 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2706
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034871 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[

## Tuning with RandomizedSearchCV

In [21]:
# Define the broad random distribution
param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'learning_rate': np.linspace(0.01, 0.2, 20),
    'max_depth': [3, 4, 5, 6, 7, 8],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.8, 1.0],         
    'colsample_bytree': [0.6, 0.8, 1.0]   
}
lgbm_random = lgb.LGBMClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=lgbm_random,
    param_distributions=param_dist,
    n_iter=10,            
    cv=2,                 
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)
y_pred_random = random_search.best_estimator_.predict(X_test)
print("\nBest RandomizedSearchCV Parameters:", random_search.best_params_)
print("RandomizedSearchCV Accuracy:", accuracy_score(y_test, y_pred_random))
print("\nRandomizedSearchCV Classification Report:")
print(classification_report(y_test, y_pred_random))


Fitting 2 folds for each of 10 candidates, totalling 20 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.110375 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 252000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968953
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032942 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2705
[LightGBM] [Info] Number of data points in the train set: 252000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968953
[LightGBM] [Warning] No further spl

## Tuning with Optuna

In [22]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }
    
    lgbm_optuna = lgb.LGBMClassifier(**param)
    lgbm_optuna.fit(X_train, y_train)
    y_pred = lgbm_optuna.predict(X_test)
    return accuracy_score(y_test, y_pred)
study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING) 
study.optimize(objective, n_trials=10) 
print("\nBest Optuna Parameters:", study.best_params)

# Fit one final model using Optuna's best params to get the classification report
lgbm_optuna_best = lgb.LGBMClassifier(**study.best_params, random_state=42)
lgbm_optuna_best.fit(X_train, y_train)
y_pred_optuna = lgbm_optuna_best.predict(X_test)
print("Optuna Top Accuracy:", accuracy_score(y_test, y_pred_optuna))
print("\nOptuna Classification Report:")
print(classification_report(y_test, y_pred_optuna))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003472 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2702
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002416 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2702
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Star

In [23]:
# load the test dataset
test_df = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/test.csv')

submission_ids = test_df['id']
X_test_final = test_df.drop(['id'], axis=1)

for col in categorical_cols:
    X_test_final[col] = X_test_final[col].astype('category')

final_predictions = lgbm_optuna_best.predict(X_test_final)

submission = pd.DataFrame({
    'id': submission_ids,
    'Irrigation_Need': final_predictions
})

# Save the DataFrame to a CSV file 
submission_path = '/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-2/lightgbm_submission.csv'
submission.to_csv(submission_path, index=False)

